In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Task 1 Data Gathering and Combination.

conn = sqlite3.connect('Library Database')

members_df = pd.read_sql_query("SELECT * FROM members", conn)
checkouts_df = pd.read_sql_query("SELECT * FROM checkouts", conn)
books_db_df = pd.read_sql_query("SELECT * FROM books", conn)

member_counts = checkouts_df.groupby('member_id').size().reset_index(name='total_borrowed')
members_updated = pd.merge(members_df, member_counts, on='member_id', how='left')
members_updated['total_borrowed'] = members_updated['total_borrowed'].fillna(0)
stage1_df = pd.merge(checkouts_df, members_updated, on='member_id', how='left')

books_json_df = pd.read_json('Book Catalog')
stage2_df = pd.merge(stage1_df, books_db_df, on='book_id', how='left')
stage2_df = pd.merge(stage2_df, books_json_df, on='book_id', how='left')

html_tables = pd.read_html('Reading Kickoff Signups')
kickoff_df = html_tables[0]
kickoff_df = kickoff_df.rename(columns={
    'Member ID': 'member_id',
    'Book ID': 'book_id',
    'Checkout Date': 'checkout_date'
})

kickoff_combined = pd.merge(kickoff_df, members_updated, on='member_id', how='left')
kickoff_combined = pd.merge(kickoff_combined, books_db_df, on='book_id', how='left')
kickoff_combined = pd.merge(kickoff_combined, books_json_df, on='book_id', how='left')

final_task1_df = pd.concat([stage2_df, kickoff_combined], ignore_index=True)

final_task1_df.to_csv('task1_combined_data.csv', index=False)


Reinitialized existing Git repository in /content/.git/
On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	31009151503778-Library-git_log.txt
	Book Catalog
	Library Database
	Reading Kickoff Signups
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
# Task 2 Data Integrity.

cleaned_df = final_task1_df.copy()

# Problem 1: Missing Values
cleaned_df['return_date'] = cleaned_df['return_date'].fillna('Not Due Yet')
cleaned_df['publication_year'] = cleaned_df['publication_year'].fillna(0)

# Problem 2:duplicate records
cleaned_df = cleaned_df.drop_duplicates()

# Problem 3: The Same Value Written Different Ways
cleaned_df['neighborhood'] = cleaned_df['neighborhood'].astype(str).str.title().str.strip()
cleaned_df['membership_status'] = cleaned_df['membership_status'].astype(str).str.title().str.strip()
cleaned_df['neighborhood'] = cleaned_df['neighborhood'].replace('Nan', np.nan)
cleaned_df['membership_status'] = cleaned_df['membership_status'].replace('Nan', np.nan)

# Problem 4: Checkouts With No Matching Member
cleaned_df = cleaned_df.dropna(subset=['first_name'])

cleaned_df.to_csv('task2_cleaned_data.csv', index=False)


On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	31009151503778-Library-git_log.txt
	Book Catalog
	Library Database
	Reading Kickoff Signups
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
# Task 3: Data Fairness and Version Control.

fairness_df = cleaned_df.groupby('neighborhood').agg(
    total_checkouts=('checkout_id', 'count'),
    unique_members=('member_id', 'nunique')
).reset_index()

fairness_df['checkouts_per_member'] = fairness_df['total_checkouts'] / fairness_df['unique_members']
print(fairness_df)

  neighborhood  total_checkouts  unique_members  checkouts_per_member
0   Heliopolis               84              13              6.461538
1        Maadi              106              20              5.300000
2    Nasr City               97              16              6.062500
3       Shubra               34               5              6.800000
4      Zamalek               62              11              5.636364
[master b8e1696] Completed Task 3 Fairness Reflection
commit b8e1696a0b077647ef41312c19606ac27cff999d
Author: mooghanem <mohammedkhaledfarag@gmail.com>
Date:   Tue Aug 18 17:15:00 2026 +0000

    Completed Task 3 Fairness Reflection

commit 7a4ee7f6010ba023c6e914996813326a3247f9fe
Author: mooghanem <mohammedkhaledfarag@gmail.com>
Date:   Tue Aug 18 17:15:00 2026 +0000

    Completed Task 3 Fairness Reflection

commit 38184ea73789c0e6605e5da5cf66799fc5b63a84
Author: mooghanem <mohammedkhaledfarag@gmail.com>
Date:   Mon Aug 17 11:15:00 2026 +0000

    Completed Task 2 Data Cl